In [16]:
from mistralai.client import Mistral
import os
import pandas as pd 
import faiss
import numpy as np
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time


In [3]:
# Initialiser l'encodeur (utilisez le modèle approprié)
encoding = tiktoken.get_encoding("cl100k_base")  # Encodeur standard pour la plupart des modèles

# Calculer les tokens pour une description
def count_tokens(text):
    return len(encoding.encode(text))

In [6]:
# Load file
source_data = pd.read_json('../data/evenements-publics-openagenda_25.json')

print(f'Nombre de lignes avant filtrage {source_data.shape[0]}')

# Filtrer les lignes de source_data avec des descriptions vide ou avec uniquement des espaces
source_data = source_data[source_data["longdescription_fr"].apply(lambda x: isinstance(x, str) and bool(x.strip()))]

print(f'Nombre de lignes après filtrage {source_data.shape[0]}')

long_desc_list = source_data["longdescription_fr"].tolist()

print(f'Nombre de lignes dans long_desc_list {len(long_desc_list)}')


Nombre de lignes avant filtrage 6819
Nombre de lignes après filtrage 6666
Nombre de lignes dans long_desc_list 6666


In [7]:
# Analyser vos descriptions
source_data["token_count"] = source_data["longdescription_fr"].apply(count_tokens)

# Statistiques
print(f"Nombre moyen de tokens: {source_data['token_count'].mean():.2f}")
print(f"Nombre max de tokens: {source_data['token_count'].max()}")
print(f"Nombre min de tokens: {source_data['token_count'].min()}")

# Afficher la distribution
print("\nDistribution des tokens:")
print(source_data["token_count"].describe())

# Voir les descriptions les plus longues
print("\nTop 5 descriptions les plus longues:")
print(source_data.nlargest(5, "token_count")[["uid", "longdescription_fr", "token_count"]])

Nombre moyen de tokens: 185.09
Nombre max de tokens: 3509
Nombre min de tokens: 7

Distribution des tokens:
count    6666.000000
mean      185.093759
std       188.703781
min         7.000000
25%        81.000000
50%       135.000000
75%       235.000000
max      3509.000000
Name: token_count, dtype: float64

Top 5 descriptions les plus longues:
           uid                                 longdescription_fr  token_count
163    2467092  <h2>Venez vous amuser, découvrir, essayer, adm...         3509
348   47974916  <p>Les rendez-vous des jardins, c'est un diman...         2791
2265  45234349  <h3><strong>Quantiques des cantiques</strong><...         2712
3758  32615579  <p>La fête des trous est de retour pour une de...         2498
3043  40172904  <p>Les rendez-vous des jardins, c'est un diman...         2430


## Chunking

Après analyse des données, le constat est le suivant : 
- 75% des descriptions < 235 tokens → Pas besoin de chunking
- Max 3509 tokens → Quelques descriptions très longues nécessitent un découpage
- Moyenne 185 tokens → Largement sous la limite de 512 tokens

Conclusion, nous allons utiliser le chunking conditionnel en découpant uniquement les descriptions qui sont supérieures à 512 tokens.

In [11]:
# Configuration du splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,  # Taille cible en tokens (marge de sécurité)
    chunk_overlap=50,  # Chevauchement pour garder le contexte
    length_function=count_tokens,
    separators=["\n\n", "\n", ". ", " ", ""]  # Découpe intelligente
)

def process_description(row):
    """Découpe uniquement si nécessaire"""
    text = row["longdescription_fr"]
    token_count = row["token_count"]
    uid = row["uid"]
    
    if token_count <= 512:
        # Pas de chunking nécessaire
        return [{
            "uid": uid,
            "chunk_id": 0,
            "text": text,
            "token_count": token_count
        }]
    else:
        # Chunking pour les longues descriptions
        chunks = text_splitter.split_text(text)
        return [{
            "uid": uid,
            "chunk_id": i,
            "text": chunk,
            "token_count": count_tokens(chunk)
        } for i, chunk in enumerate(chunks)]

# Appliquer le traitement
chunks_list = []
for _, row in source_data.iterrows():
    chunks_list.extend(process_description(row))

# Créer un nouveau DataFrame avec les chunks
chunks_df = pd.DataFrame(chunks_list)

print(f"Nombre de descriptions originales: {len(source_data)}")
print(f"Nombre total de chunks: {len(chunks_df)}")
print(f"Descriptions découpées: {len(chunks_df[chunks_df['chunk_id'] > 0]['uid'].unique())}")

Nombre de descriptions originales: 6666
Nombre total de chunks: 7242
Descriptions découpées: 271


## Embedding
Utilisation de Mistral pour l'embedding

In [ ]:
def create_smart_batches(texts, max_tokens_per_batch=8000):
    """Crée des batches en fonction du nombre de tokens"""
    batches = []
    current_batch = []
    current_tokens = 0
    
    for text in texts:
        text_tokens = count_tokens(text)
        
        if current_tokens + text_tokens > max_tokens_per_batch and current_batch:
            batches.append(current_batch)
            current_batch = [text]
            current_tokens = text_tokens
        else:
            current_batch.append(text)
            current_tokens += text_tokens
    
    if current_batch:
        batches.append(current_batch)
    
    return batches

In [17]:
texts_to_embed = chunks_df["text"].tolist()

batches = create_smart_batches(texts_to_embed, max_tokens_per_batch=7000)
print(f"Nombre de batches créés: {len(batches)}")

all_embeddings = []

with Mistral(
    api_key=os.getenv("MISTRAL_API_KEY", ""),
) as mistral:
    
    for i, batch in enumerate(batches):
        print(f"Batch {i+1}/{len(batches)} - {len(batch)} textes")
        
        try:
            res = mistral.embeddings.create(
                model="mistral-embed", 
                inputs=batch
            )
            
            all_embeddings.extend([item.embedding for item in res.data])
            print(f"  ✓ Total embeddings accumulés: {len(all_embeddings)}")
            
            # Pause entre les batches pour respecter le rate limit
            if i < len(batches) - 1:  # Pas de pause après le dernier batch
                wait_time = 2  # Secondes entre chaque batch
                print(f"  ⏳ Pause de {wait_time}s...")
                time.sleep(wait_time)
                
        except Exception as e:
            if "429" in str(e) or "rate_limited" in str(e).lower():
                print(f"  ⚠️ Rate limit atteint, pause de 60s...")
                time.sleep(60)
                # Réessayer le batch
                print(f"  🔄 Nouvelle tentative pour le batch {i+1}")
                res = mistral.embeddings.create(
                    model="mistral-embed", 
                    inputs=batch
                )
                all_embeddings.extend([item.embedding for item in res.data])
                print(f"  ✓ Batch traité après retry")
            else:
                print(f"  ✗ Erreur: {e}")
                raise

# Vérification avant assignation
print(f"\nVérification:")
print(f"  - Nombre de chunks: {len(chunks_df)}")
print(f"  - Nombre d'embeddings: {len(all_embeddings)}")

# Ajouter les embeddings au DataFrame de chunks
chunks_df["embedding"] = all_embeddings

# Sauvegarder les métadonnées pour la recherche
chunks_df.to_json("chunks_with_embeddings.json", orient="records", force_ascii=False)

Nombre de batches créés: 181
Batch 1/181 - 46 textes
  ✓ Total embeddings accumulés: 46
  ⏳ Pause de 2s...
Batch 2/181 - 30 textes
  ✓ Total embeddings accumulés: 76
  ⏳ Pause de 2s...
Batch 3/181 - 38 textes
  ✓ Total embeddings accumulés: 114
  ⏳ Pause de 2s...
Batch 4/181 - 45 textes
  ✓ Total embeddings accumulés: 159
  ⏳ Pause de 2s...
Batch 5/181 - 30 textes
  ✓ Total embeddings accumulés: 189
  ⏳ Pause de 2s...
Batch 6/181 - 30 textes
  ✓ Total embeddings accumulés: 219
  ⏳ Pause de 2s...
Batch 7/181 - 48 textes
  ✓ Total embeddings accumulés: 267
  ⏳ Pause de 2s...
Batch 8/181 - 52 textes
  ✓ Total embeddings accumulés: 319
  ⏳ Pause de 2s...
Batch 9/181 - 35 textes
  ✓ Total embeddings accumulés: 354
  ⏳ Pause de 2s...
Batch 10/181 - 31 textes
  ✓ Total embeddings accumulés: 385
  ⏳ Pause de 2s...
Batch 11/181 - 31 textes
  ✓ Total embeddings accumulés: 416
  ⏳ Pause de 2s...
Batch 12/181 - 30 textes
  ✓ Total embeddings accumulés: 446
  ⏳ Pause de 2s...
Batch 13/181 - 42 text

In [ ]:
# Initialisation de l'index Faiss
embeddings_array = np.array([item for item in chunks_df["embedding"]])
dimension = embeddings_array.shape[1]  # Nombre de features
n_vectors = embeddings_array.shape[0]
print(f"Dimension des vecteurs: {dimension}")
print(f"Nombre de vecteurs à indexer: {n_vectors}")

index = faiss.IndexFlatL2(dimension)

# Ajout des embeddings à l'index
index.add(embeddings_array)

# Sauvegarde de l'index sur le disque
faiss.write_index(index, "faiss_index.idx")

print(f"Index FAISS sauvegardé avec {index.ntotal} vecteurs")

Dimension des vecteurs: 1024
Nombre de vecteurs à indexer: 7242
Index FAISS sauvegardé avec 7242 vecteurs


In [19]:
# Exemple de recherche
def search_events(query, k=5):
    """Recherche les k événements les plus similaires"""
    
    # 1. Vectoriser la requête
    with Mistral(api_key=os.getenv("MISTRAL_API_KEY", "")) as mistral:
        query_res = mistral.embeddings.create(
            model="mistral-embed",
            inputs=[query]
        )
        query_embedding = np.array([query_res.data[0].embedding])
    
    # 2. Rechercher dans FAISS
    distances, indices = index.search(query_embedding, k)
    
    # 3. Récupérer les résultats
    results = chunks_df.iloc[indices[0]]
    
    return results[["uid", "chunk_id", "text", "token_count"]], distances[0]

# Test
results, distances = search_events("concert de musique classique", k=5)
print("Top 5 événements similaires:")
print(results)
print(f"\nDistances: {distances}")

Top 5 événements similaires:
           uid  chunk_id                                               text  \
5239  32732635         0  <p>Concert acoustique de musique trad irlandai...   
2190  90148048         0  <p><strong>Concert à Aix-en-Provence : Shostak...   
5092  55339912         0  <h3><strong>Un concert éclatant à Nice 🌸💃</str...   
45    85623782         0  <p>Venez tester vos connaissances en musique d...   
6507  74877992         0  <p>Ouverture de l’établissement de 10h à 12h e...   

      token_count  
5239           16  
2190          349  
5092          373  
45             25  
6507          286  

Distances: [0.4542258  0.45462716 0.4599963  0.47203267 0.47652996]
